In [1]:
import tensorflow as tf
from tensorflow import keras
import os
import pandas as pd
from tensorflow.keras.applications import ResNet50V2
from sklearn.model_selection import StratifiedKFold
import numpy as np
import matplotlib.pyplot as plt

In [2]:
PATH = 'datasets/ISIC_2020_corrected'
os.listdir(PATH)

['multiclass',
 'subset.csv',
 'test_split.csv',
 'train',
 'train.csv',
 'train_resized',
 'train_sd_split.csv',
 'train_split.csv',
 'val_split.csv']

In [3]:
train = pd.read_csv(f'{PATH}/train_split.csv')
val = pd.read_csv(f'{PATH}/val_split.csv')
test = pd.read_csv(f'{PATH}/test_split.csv')
train.shape, val.shape

((33126, 9), (4969, 9))

In [4]:
train.head()

,image_name,patient_id,lesion_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target
0,ISIC_2637011,IP_7279968,IL_7972535,male,45.0,head/neck,unknown,benign,0
1,ISIC_0015719,IP_3075186,IL_4649854,female,45.0,upper extremity,unknown,benign,0
2,ISIC_0052212,IP_2842074,IL_9087444,female,50.0,lower extremity,nevus,benign,0
3,ISIC_0068279,IP_6890425,IL_4255399,female,45.0,head/neck,unknown,benign,0
4,ISIC_0074268,IP_8723313,IL_6898037,female,55.0,upper extremity,unknown,benign,0


In [5]:
BATCH_SIZE = 16
AUTO = tf.data.experimental.AUTOTUNE

def decode(name, label):
    img = tf.io.read_file(name)
    img = tf.image.decode_jpeg(img, channels = 3)
    img = tf.cast(img, tf.float32)
    return img, label
    
def load_ds(df):
    options = tf.data.Options()
    options.experimental_deterministic = False
    imgs, labels = df["image_name"].values, df["target"].values
    imgs = [f'{PATH}/train_resized/{name}.jpg' for name in imgs]
    ds = tf.data.Dataset.from_tensor_slices((imgs, labels))
    ds = ds.with_options(options)
    ds = ds.map(decode, num_parallel_calls=AUTO)
    ds = ds.cache()
    ds = ds.shuffle(2048)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(buffer_size=AUTO)
    return ds

In [6]:
train_ds = load_ds(train)
val_ds = load_ds(val)
test_ds = load_ds(test)

In [7]:
IMAGE_SIZE = (256, 256, 3)

encoder = ResNet50V2(
    include_top=False,
    input_shape=IMAGE_SIZE,
    weights='imagenet'
)
encoder.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE)
x = keras.layers.Rescaling(1./255)(inputs)
x = encoder(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 256, 256, 3)]     0         
                                                                 
 rescaling (Rescaling)       (None, 256, 256, 3)       0         
                                                                 
 resnet50v2 (Functional)     (None, 8, 8, 2048)        23564800  
                                                                 
 global_average_pooling2d (G  (None, 2048)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 1)                 2049      
                                                                 
Total params: 23,566,849
Trainable params: 2,049
Non-trainable params: 23,564,800
_____________________________________________

In [8]:
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.AUC(name="auc", curve='PR'), keras.metrics.FalseNegatives(), keras.metrics.TruePositives()]
)

In [9]:
filepath = 'models/checkpoints/baseline2PR'
cb = tf.keras.callbacks.ModelCheckpoint(
    filepath = filepath,
    monitor="val_auc",
    verbose=0,
    save_best_only=True,
    save_weights_only=True,
    mode="max"
)

In [10]:
history = model.fit(train_ds,
                    epochs=10,
                    validation_data=val_ds,
                    #validation_steps=10,
                    callbacks=[cb]
                   )

Epoch 1/10
2071/2071 [==============================] - 115s 52ms/step - loss: 0.0851 - auc: 0.0651 - false_negatives: 576.0000 - true_positives: 8.0000 - val_loss: 0.0652 - val_auc: 0.2658 - val_false_negatives: 79.0000 - val_true_positives: 9.0000
Epoch 2/10
2071/2071 [==============================] - 101s 49ms/step - loss: 0.0723 - auc: 0.1462 - false_negatives: 561.0000 - true_positives: 23.0000 - val_loss: 0.0604 - val_auc: 0.3210 - val_false_negatives: 82.0000 - val_true_positives: 6.0000
Epoch 3/10
2071/2071 [==============================] - 104s 50ms/step - loss: 0.0690 - auc: 0.1757 - false_negatives: 552.0000 - true_positives: 32.0000 - val_loss: 0.0582 - val_auc: 0.3808 - val_false_negatives: 79.0000 - val_true_positives: 9.0000
Epoch 4/10
2071/2071 [==============================] - 109s 53ms/step - loss: 0.0659 - auc: 0.2248 - false_negatives: 541.0000 - true_positives: 43.0000 - val_loss: 0.0548 - val_auc: 0.4064 - val_false_negatives: 74.0000 - val_true_positives: 14.0

In [11]:
history.history

{'loss': [0.08508007228374481,
  0.07229488343000412,
  0.06898795813322067,
  0.06593244522809982,
  0.0643431693315506,
  0.06270713359117508,
  0.06129639968276024,
  0.05965305492281914,
  0.05875527486205101,
  0.058001454919576645],
 'auc': [0.06513607501983643,
  0.14620301127433777,
  0.17572571337223053,
  0.22478435933589935,
  0.23381435871124268,
  0.25844645500183105,
  0.2802999019622803,
  0.310708612203598,
  0.3273889124393463,
  0.32562771439552307],
 'false_negatives': [576.0,
  561.0,
  552.0,
  541.0,
  537.0,
  533.0,
  523.0,
  500.0,
  503.0,
  509.0],
 'true_positives': [8.0, 23.0, 32.0, 43.0, 47.0, 51.0, 61.0, 84.0, 81.0, 75.0],
 'val_loss': [0.06521190702915192,
  0.06037186458706856,
  0.058224719017744064,
  0.05482752248644829,
  0.05729106441140175,
  0.05025554448366165,
  0.05167210474610329,
  0.049128491431474686,
  0.0546482689678669,
  0.049933552742004395],
 'val_auc': [0.26580578088760376,
  0.3209833800792694,
  0.3808401823043823,
  0.4064469039

In [1]:
auc = history.history['auc']
val_auc = history.history['val_auc']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(10)

plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, auc, label='Training auc')
plt.plot(epochs_range, val_auc, label='Validation auc')
plt.legend(loc='lower right')
plt.title('Training and Validation Area Under Curve')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

NameError: name 'history' is not defined

In [15]:
model.load_weights(filepath)
results = model.evaluate(test_ds)
print(results)

311/311 [==============================] - 13s 40ms/step - loss: 0.0522 - auc: 0.4792 - false_negatives: 83.0000 - true_positives: 4.0000
[0.052171848714351654, 0.47921937704086304, 83.0, 4.0]


In [21]:
history.history['evaluation'] = results
history.history

{'loss': [0.12608055770397186,
  0.08349297195672989,
  0.11130350083112717,
  0.08301620930433273,
  0.07850052416324615,
  0.07594470679759979,
  0.0754527822136879,
  0.07115525007247925,
  0.06950052082538605,
  0.06650582700967789],
 'auc': [0.6173259019851685,
  0.7649521827697754,
  0.7744652032852173,
  0.7297800779342651,
  0.7993853688240051,
  0.8074852228164673,
  0.8147390484809875,
  0.8655700087547302,
  0.8577432632446289,
  0.8945533633232117],
 'false_negatives': [41.0,
  41.0,
  41.0,
  41.0,
  41.0,
  41.0,
  41.0,
  41.0,
  41.0,
  41.0],
 'true_positives': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 'val_loss': [0.09472941607236862,
  0.084307461977005,
  0.059485454112291336,
  0.08389177918434143,
  0.08924411237239838,
  0.09721383452415466,
  0.10287590324878693,
  0.08853951841592789,
  0.08961668610572815,
  0.09985748678445816],
 'val_auc': [0.7143710255622864,
  0.8392221331596375,
  0.6352968215942383,
  0.7494580149650574,
  0.7030917406082153,


In [16]:

model.save('models/baseline2PR.h5')